In [ ]:
%pip install langchain-core langchain-google-genai langchain-community langgraph tavily-python python-dotenv

In [ ]:
import os

required = ("GOOGLE_API_KEY", "TAVILY_API_KEY")
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Set these environment variables before running: " + ", ".join(missing))


In [ ]:
from typing import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI # استدعاء مكتبة جمناي
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 1. State Graph
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 2. Tool Calling
search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]
tool_node = ToolNode(tools)

# 3. LLM & Tools (استخدام نموذج جمناي)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# 4. Nodes
def call_model(state: AgentState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# 5. Conditional Routing
def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "action"
    return "end"

# 6. Graph Building
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"action": "action", "end": END})
workflow.add_edge("action", "agent")

# 7. Memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# 8. Execution
def main():
    config = {"configurable": {"thread_id": "research_session_02"}}
    query = "Search for information about 'Agentic AI' and 'LangGraph', then provide a summarized response about how they work together."
    
    print(f"User Query: {query}\n")
    print("Agent is thinking and searching...\n" + "-"*50)

    inputs = {"messages": [HumanMessage(content=query)]}
    
    for event in app.stream(inputs, config=config, stream_mode="values"):
        event["messages"][-1].pretty_print()

if __name__ == "__main__":
    main()